In [6]:
!pip install -q -U transformers accelerate

In [7]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())

CUDA: True
GPUs: 2


In [8]:
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
import torch

MODEL_ID = "Qwen/Qwen2.5-Omni-3B"

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)


print("Qwen loaded!")

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

Qwen loaded!


In [9]:
import torch

def moodmix_qwen(task, content):

    prompts = {
        "lyrics_analysis": f"""
        Analyse these lyrics and return JSON with:
        themes, emotional_tone, sentiment, genres,
        instruments, listening_mood, explanation.

        Lyrics: {content}
        """,

        "search_understanding": f"""
        Convert this music search request into JSON with:
        search_query, genres, moods, tempo_preference.

        Request: {content}
        """,

        "prompt_enhancement": f"""
        Convert this music generation request into JSON with:
        enhanced_prompt, genre, mood, instruments, tempo.

        Request: {content}
        """,

        "recommendation_explanation": f"""
        Explain why these music recommendations match the user.
        Return JSON with explanation, mood_match, genre_match.

        Context: {content}
        """
    }

    if task not in prompts:
        raise ValueError(f"Unknown task: {task}")

    conversation = [{
        "role": "user",
        "content": [{"type": "text", "text": prompts[task]}]
    }]

    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            return_audio=False
        )

    input_length = inputs["input_ids"].shape[1]
    generated = output[:, input_length:]

    result = processor.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return result[0]


print("MoodMix Qwen functions ready!")

MoodMix Qwen functions ready!


In [10]:
result = moodmix_qwen(
    "lyrics_analysis",
    "Walking through the city lights, thinking about the days gone by."
)

print(result)

```json
{
  "themes": ["reflection", "memories", "urban life"],
  "emotional_tone": "reflective",
  "sentiment": "neutral",
  "genres": ["soul", "rnb", "pop"],
  "instruments": ["piano", "guitar", "drums", "vocals"],
  "listening_mood": "reflective",
  "explanation": "The lyrics evoke a sense of nostalgia and introspection, as the speaker walks through the city lights and reflects on past days. The emotional tone is reflective, and the sentiment is neutral. The genres include soul, R&B, and pop, which are often used to convey personal stories and emotions. The instruments used are piano, guitar, drums, and vocals, which are commonly used in these genres to create a melodic and emotional atmosphere. Listening to this song in a reflective mood can help listeners connect with the speaker's emotions and thoughts."
}
```


In [11]:
tests = {
    "lyrics_analysis": "I miss the days when we were young and free.",
    "search_understanding": "Find energetic Hindi rock songs for working out.",
    "prompt_enhancement": "Generate calm piano music for a rainy evening.",
    "recommendation_explanation": "The user likes acoustic music, soft vocals and reflective songs."
}

for task, content in tests.items():
    print(f"\n--- {task} ---")
    print(moodmix_qwen(task, content))


--- lyrics_analysis ---
```json
{
  "themes": ["youth", "freedom", "memories"],
  "emotional_tone": "nostalgic",
  "sentiment": "positive",
  "genres": ["pop", "soul", "rock"],
  "instruments": ["guitar", "piano", "drums"],
  "listening_mood": "reflective",
  "explanation": "The lyrics express a longing for simpler times when the speaker was young and free. The nostalgic tone suggests a sense of loss or yearning for past experiences. The use of instruments like guitar, piano, and drums can create a melodic and emotional atmosphere, enhancing the reflective mood of the song."
}
```

--- search_understanding ---
```json
{
  "search_query": "energetic Hindi rock songs for working out",
  "genres": ["Hindi rock"],
  "moods": ["energetic"],
  "tempo_preference": "working out"
}
```

--- prompt_enhancement ---
```json
{
  "enhanced_prompt": "Create a serene piano piece that evokes the feeling of a peaceful rainy evening.",
  "genre": "Classical",
  "mood": "Calm",
  "instruments": "Piano",


In [12]:
!pip -q install fastapi uvicorn pyngrok

In [13]:
# Authenticate ngrok using Kaggle Secrets
# Secret Name: NGROK_AUTHTOKEN (Configured via Kaggle Notebook -> Add-ons -> Secrets)
import sys
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
except Exception as e:
    print("CRITICAL ERROR: Failed to retrieve NGROK_AUTHTOKEN from Kaggle Secrets.")
    print("Action Required: Please add your ngrok authentication token under Kaggle Notebook -> Add-ons -> Secrets with label NGROK_AUTHTOKEN.")
    print(f"Details: {e}")
    sys.exit(1)

if not ngrok_token or not str(ngrok_token).strip():
    print("CRITICAL ERROR: NGROK_AUTHTOKEN secret is empty.")
    print("Action Required: Provide a valid ngrok authentication token in Kaggle Secrets.")
    sys.exit(1)

ngrok.set_auth_token(str(ngrok_token).strip())

# Connect ngrok tunnel
public_url = ngrok.connect(8000).public_url
print("Tunnel established successfully!")
print(f"Public Tunnel URL: {public_url}")


https://valiant-program-parched.ngrok-free.dev                                                      


In [21]:

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import threading
import json

app = FastAPI(title="MoodMix Qwen API")


class AnalyzeRequest(BaseModel):
    task: str
    content: str


# Supported Qwen tasks
ALLOWED_TASKS = [
    "lyrics_analysis",
    "search_understanding",
    "voice_query_understanding",
    "prompt_enhancement",
    "recommendation_explanation",
    "discover_personalization",
    "music_analysis",
    "listening_preference_analysis",
    "ai_dj_transition"
]


@app.get("/")
def health():
    return {
        "status": "online",
        "model": "Qwen2.5-Omni-3B",
        "supported_tasks": ALLOWED_TASKS
    }


@app.post("/analyze")
def analyze(request: AnalyzeRequest):

    if request.task not in ALLOWED_TASKS:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid task. Supported tasks: {ALLOWED_TASKS}"
        )

    if not request.content.strip():
        raise HTTPException(
            status_code=400,
            detail="Content cannot be empty"
        )

    try:
        result = moodmix_qwen(
            request.task,
            request.content
        )

        return {
            "task": request.task,
            "result": result,
            "status": "success"
        }

    except Exception as e:
        print(f"Qwen error [{request.task}]: {str(e)}")

        raise HTTPException(
            status_code=500,
            detail="Qwen inference failed"
        )


def start_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

print("MoodMix Qwen API started!")
print("Supported tasks:", ALLOWED_TASKS)


MoodMix Qwen API started!
Supported tasks: ['lyrics_analysis', 'search_understanding', 'voice_query_understanding', 'prompt_enhancement', 'recommendation_explanation', 'discover_personalization', 'music_analysis', 'listening_preference_analysis', 'ai_dj_transition']


INFO:     Started server process [314]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [22]:
from pyngrok import ngrok

public_url = ngrok.connect(8000).public_url
print("YOUR QWEN API:", public_url)
print("TEST ENDPOINT:", public_url + "/docs")

YOUR QWEN API: https://valiant-program-parched.ngrok-free.dev
TEST ENDPOINT: https://valiant-program-parched.ngrok-free.dev/docs


In [23]:
def understand_search(query):
    prompt = f"""
    Understand this music search query.
    Extract the user's intent and preferences.

    Return ONLY valid JSON:
    {{
      "original_query": "{query}",
      "search_query": "",
      "artist": null,
      "genre": [],
      "mood": [],
      "language": null,
      "tempo": null,
      "intent": "",
      "keywords": []
    }}

    Do not invent artists or songs.
    Query: {query}
    """
    return moodmix_qwen("search_understanding", prompt)

In [17]:
def understand_voice_query(transcript):
    prompt = f"""
    Interpret this speech-recognized music query.
    Correct obvious speech recognition errors where context allows.

    Return ONLY valid JSON:
    {{
      "original_transcript": "{transcript}",
      "corrected_query": "",
      "intent": "",
      "genre": [],
      "mood": [],
      "artist": null,
      "keywords": []
    }}

    Do not invent artists or songs.
    Transcript: {transcript}
    """
    return moodmix_qwen("voice_query_understanding", prompt)

In [18]:
def interpret_music_analysis(audio_metrics):
    prompt = f"""
    Interpret these measured audio-analysis results.

    AUDIO METRICS:
    {audio_metrics}

    Return ONLY valid JSON:
    {{
      "overall_description": "",
      "detected_mood": [],
      "musical_character": "",
      "energy_interpretation": "",
      "listening_context": "",
      "explanation": ""
    }}

    Do not change or invent BPM, key, energy or other measurements.
    Interpret only the supplied metrics.
    """
    return moodmix_qwen("music_analysis", prompt)

In [19]:
def analyze_library(recent_tracks, liked_tracks):
    prompt = f"""
    Analyse the user's actual music library.

    RECENTLY PLAYED:
    {recent_tracks}

    LIKED TRACKS:
    {liked_tracks}

    Return ONLY valid JSON:
    {{
      "listening_personality": "",
      "preferred_genres": [],
      "preferred_moods": [],
      "listening_patterns": [],
      "recommendation_direction": "",
      "insights": []
    }}

    Base all conclusions on the supplied tracks.
    Do not invent listening history or preferences.
    """
    return moodmix_qwen("listening_preference_analysis", prompt)

In [20]:
def analyze_dj_transition(track_a, track_b):
    prompt = f"""
    Evaluate a possible transition between two tracks.

    TRACK A:
    {track_a}

    TRACK B:
    {track_b}

    Return ONLY valid JSON:
    {{
      "transition_suitability": "",
      "energy_compatibility": "",
      "tempo_compatibility": "",
      "mood_compatibility": "",
      "transition_suggestion": "",
      "explanation": ""
    }}

    Use only the supplied metadata.
    Do not invent BPM or musical properties.
    """
    return moodmix_qwen("ai_dj_transition", prompt)